# Emergency Analysis & Flight Path Map
Flags emergency squawk codes and renders interactive flight tracks with Folium.

## 1. Import & Load Data

In [26]:
import os
import sys
import glob
import pandas as pd
import folium
import numpy as np

# ── FILE SELECTION ────────────────────────────────────────────────────────────
# Set SPECIFIC_FILE to load a particular day, or leave as None to auto-load
# the most recent daily CSV.
#
# Examples:
#   SPECIFIC_FILE = "2026-05-27"          # just the date
#   SPECIFIC_FILE = "data/2026-05/2026-05-27.csv"  # relative path
#   SPECIFIC_FILE = None                  # auto-load latest (default)
# ─────────────────────────────────────────────────────────────────────────────
SPECIFIC_FILE = None

# Robust project root detection
cwd = os.getcwd()
project_root = next(
    (os.path.abspath(p) for p in [cwd, os.path.join(cwd, '..'), os.path.join(cwd, '..', '..')]
     if os.path.exists(os.path.join(p, 'mapping.py'))),
    cwd
)
sys.path.insert(0, project_root)
from mapping import MILITARY_BASES

# Resolve the CSV path
if SPECIFIC_FILE:
    # Accept bare date (YYYY-MM-DD), relative path, or absolute path
    sf = SPECIFIC_FILE.strip()
    if len(sf) == 10 and sf[4] == '-' and sf[7] == '-':          # bare date
        sf = os.path.join('data', sf[:7], f'{sf}.csv')
    target = sf if os.path.isabs(sf) else os.path.join(project_root, sf)
    if not os.path.exists(target):
        available = sorted(glob.glob(os.path.join(project_root, 'data', '[0-9][0-9][0-9][0-9]-[0-9][0-9]', '*.csv')))
        print("Available files:")
        for f in available:
            print(f"  {os.path.relpath(f, project_root)}")
        raise FileNotFoundError(f"File not found: {os.path.relpath(target, project_root)}")
    latest_csv = target
else:
    daily_csvs = sorted(glob.glob(os.path.join(project_root, 'data', '[0-9][0-9][0-9][0-9]-[0-9][0-9]', '*.csv')))
    if not daily_csvs:
        raise FileNotFoundError("No daily CSV files found under data/YYYY-MM/")
    latest_csv = daily_csvs[-1]

print(f"Loading: {os.path.relpath(latest_csv, project_root)}")

df = pd.read_csv(latest_csv)
df['datetime'] = pd.to_datetime(df['datetime'])
df['squawk'] = df['squawk'].astype(str).str.strip()
df = df.sort_values(['hex', 'datetime']).reset_index(drop=True)

print(f"Total records:     {len(df)}")
print(f"Unique aircraft:   {df['hex'].nunique()}")
print(f"Unique callsigns:  {df['callsign'].nunique()}")
print(f"Time range:        {df['datetime'].min()} → {df['datetime'].max()}")
df.head()

Loading: data/2026-08/2026-08-09.csv
Total records:     3726
Unique aircraft:   50
Unique callsigns:  30
Time range:        2026-08-09 00:46:44.127489 → 2026-08-09 03:46:58.188991


,hex,datetime,callsign,tail number,squawk,altitude,latitude,longitude,type,heading,ground speed,vertical rate,emergency,source
0,06a255,2026-08-09 00:46:44.127574,LHOB280,A7-MAC,2052,34000,39.734299,10.940114,C-17A Globemaster,287.0,438.0,-192.0,False,adsb_icao
1,06a255,2026-08-09 00:47:44.432440,LHOB280,A7-MAC,2052,34000,39.771332,10.791381,C-17A Globemaster,287.0,439.0,-192.0,False,adsb_icao
2,06a255,2026-08-09 00:48:44.718872,LHOB280,A7-MAC,2052,34000,39.809143,10.638885,C-17A Globemaster,287.0,441.0,0.0,False,adsb_icao
3,06a255,2026-08-09 00:49:45.271090,LHOB280,A7-MAC,2052,34000,39.846436,10.487488,C-17A Globemaster,287.0,443.0,-128.0,False,adsb_icao
4,06a255,2026-08-09 00:50:45.621130,LHOB280,A7-MAC,2052,34000,39.884308,10.332343,C-17A Globemaster,287.0,442.0,-128.0,False,adsb_icao


## 1b. Data Cleaning — Remove Spoofed / Erroneous Positions

In [27]:
from geopy.distance import geodesic as geo_distance

MAX_SPEED_KNOTS = 1500

def clean_track(ac_df):
    """Drop fixes that imply physically impossible speed from the previous accepted fix."""
    ac_df = ac_df.sort_values('datetime').reset_index(drop=True)
    keep = [0]  # always keep the first fix
    for i in range(1, len(ac_df)):
        prev = ac_df.iloc[keep[-1]]
        curr = ac_df.iloc[i]
        dt_hours = (curr['datetime'] - prev['datetime']).total_seconds() / 3600
        if dt_hours <= 0:
            continue
        dist_nm = geo_distance(
            (prev['latitude'], prev['longitude']),
            (curr['latitude'], curr['longitude'])
        ).nautical
        if dist_nm / dt_hours <= MAX_SPEED_KNOTS:
            keep.append(i)
    return ac_df.iloc[keep]


valid_mask = (
    pd.to_numeric(df['latitude'],  errors='coerce').notna() &
    pd.to_numeric(df['longitude'], errors='coerce').notna()
)
df_valid = df[valid_mask].copy()
df_valid['latitude']  = pd.to_numeric(df_valid['latitude'])
df_valid['longitude'] = pd.to_numeric(df_valid['longitude'])

before = len(df_valid)

# Use a for loop — avoids pandas 3.x groupby.apply dropping the key column
cleaned = [clean_track(grp) for _, grp in df_valid.groupby('hex')]
df_clean = pd.concat(cleaned, ignore_index=True) if cleaned else df_valid.iloc[0:0].copy()

removed = before - len(df_clean)
df_no_pos = df[~valid_mask]
df = pd.concat([df_clean, df_no_pos]).sort_values(['hex', 'datetime']).reset_index(drop=True)

print(f"Removed {removed} likely spoofed/erroneous fixes ({removed/before*100:.1f}% of positioned records)")
print(f"Remaining records: {len(df)}")

Removed 543 likely spoofed/erroneous fixes (14.6% of positioned records)
Remaining records: 3183


## 2. Emergency Flag Analysis
Squawk codes: **7500** Hijacking · **7600** Radio Failure · **7700** General Emergency

In [28]:
EMERGENCY_SQUAWKS = {
    '7500': 'HIJACKING',
    '7600': 'RADIO FAILURE',
    '7700': 'GENERAL EMERGENCY'
}

# Primary: use the collector's emergency boolean; secondary: squawk cross-check
df['emergency_type'] = df['squawk'].map(EMERGENCY_SQUAWKS)
df['is_emergency'] = df['emergency'].astype(str).str.lower().isin(['true', '1']) | df['emergency_type'].notna()
# Fill emergency_type label for collector-flagged rows without a standard squawk
df.loc[df['is_emergency'] & df['emergency_type'].isna(), 'emergency_type'] = 'EMERGENCY (non-standard)'

emergencies = df[df['is_emergency']].copy()

if emergencies.empty:
    print('No emergencies detected in this dataset.')
else:
    print(f'⚠  EMERGENCIES DETECTED: {len(emergencies)} records\n')
    summary = emergencies.groupby(['callsign', 'type', 'squawk', 'emergency_type']).agg(
        first_seen=('datetime', 'min'),
        last_seen=('datetime', 'max'),
        records=('hex', 'count')
    ).reset_index()
    display(summary)

No emergencies detected in this dataset.


In [29]:
# Full emergency records detail
if not emergencies.empty:
    display(emergencies[['callsign', 'type', 'tail number',
                          'squawk', 'emergency_type', 'datetime',
                          'altitude', 'latitude', 'longitude']].reset_index(drop=True))
else:
    # Show squawk distribution as reference
    print('Top squawk codes observed in dataset:')
    display(df.groupby('squawk').size().sort_values(ascending=False).head(15).rename('count').reset_index())

Top squawk codes observed in dataset:


,squawk,count
0,1630,180
1,4015,180
2,7221,160
3,1513,154
4,6011,147
5,1577,142
6,1504,138
7,1441,135
8,6001,122
9,2550,116


## 3. Flight Path Map
Each aircraft gets a colored track. Emergency squawkers are highlighted in red. Toggle military bases via the layer control.

In [30]:
def altitude_color(alt_ft):
    """Multi-stop altitude color scale with linear interpolation between stops.
    Returns white when altitude is None or cannot be parsed."""
    if alt_ft is None:
        return '#ffffff'
    STOPS = [
        (    0, (255, 165,   0)),   # orange
        (10000, (255, 255,   0)),   # yellow
        (20000, (  0, 255,   0)),   # green
        (30000, (  0, 255, 255)),   # cyan
        (35000, (  0,   0, 255)),   # blue
        (40000, (128,   0, 128)),   # purple
        (50000, (255,   0,   0)),   # red
    ]
    try:
        a = max(0.0, min(float(alt_ft), 50000))
    except (ValueError, TypeError):
        return '#ffffff'

    for i in range(len(STOPS) - 1):
        lo_alt, lo_rgb = STOPS[i]
        hi_alt, hi_rgb = STOPS[i + 1]
        if lo_alt <= a <= hi_alt:
            t = (a - lo_alt) / (hi_alt - lo_alt)
            r = int(lo_rgb[0] + (hi_rgb[0] - lo_rgb[0]) * t)
            g = int(lo_rgb[1] + (hi_rgb[1] - lo_rgb[1]) * t)
            b = int(lo_rgb[2] + (hi_rgb[2] - lo_rgb[2]) * t)
            return f'#{r:02x}{g:02x}{b:02x}'

    return f'#{STOPS[-1][1][0]:02x}{STOPS[-1][1][1]:02x}{STOPS[-1][1][2]:02x}'


def seg_avg(series, j):
    """Average of two consecutive series values, falling back to whichever is valid."""
    a, b = series.iloc[j], series.iloc[j + 1]
    if pd.notna(a) and pd.notna(b):
        return (a + b) / 2
    return a if pd.notna(a) else b if pd.notna(b) else None


# Filter to rows with valid coordinates
map_df = df[~df['latitude'].astype(str).str.contains('UNKNOWN') &
            ~df['longitude'].astype(str).str.contains('UNKNOWN')].copy()
map_df['latitude']  = pd.to_numeric(map_df['latitude'],  errors='coerce')
map_df['longitude'] = pd.to_numeric(map_df['longitude'], errors='coerce')
map_df = map_df.dropna(subset=['latitude', 'longitude', 'hex'])

# Normalize type for grouping (NaN / empty → 'Unknown')
map_df['type_display'] = map_df['type'].fillna('Unknown').replace('', 'Unknown')

print(f"Aircraft with position data: {map_df['hex'].nunique()}")
fix_counts = map_df.groupby('hex').size()
print(f"  ≥2 fixes (will draw track): {(fix_counts >= 2).sum()}")
print(f"  1 fix  (skipped):           {(fix_counts == 1).sum()}")

type_names = sorted(map_df['type_display'].unique())
print(f"\nAircraft types in dataset ({len(type_names)}):")
for t in type_names:
    count = map_df.groupby('type_display').get_group(t)['hex'].nunique()
    print(f"  {t}: {count} aircraft")

center_lat = map_df['latitude'].mean()
center_lon = map_df['longitude'].mean()

# max_bounds=True limits panning to one copy of the world
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=4,
    tiles=None,
    max_bounds=True
)
# control=False keeps the base tile out of the LayerControl panel
folium.TileLayer('CartoDB dark_matter', no_wrap=True, control=False).add_to(m)

# --- Military Base Markers (toggleable layer) ---
base_group = folium.FeatureGroup(name='⬡ Military Bases', show=False)
for base_name, coords in MILITARY_BASES.items():
    folium.CircleMarker(
        location=[coords['lat'], coords['lon']],
        radius=4,
        color='#888888',
        fill=True,
        fill_color='#888888',
        fill_opacity=0.6,
        tooltip=base_name
    ).add_to(base_group)
base_group.add_to(m)

# --- One FeatureGroup per aircraft type (all on by default) ---
type_groups = {t: folium.FeatureGroup(name=f'✈ {t}', show=True) for t in type_names}

# --- Flight Paths ---
for hex_id, ac_df in map_df.groupby('hex'):
    ac_df = ac_df.sort_values('datetime').reset_index(drop=True)
    coords = list(zip(ac_df['latitude'], ac_df['longitude']))

    if len(coords) < 2:
        continue

    alt_series = pd.to_numeric(ac_df['altitude'],     errors='coerce')
    gs_series  = pd.to_numeric(ac_df['ground speed'], errors='coerce')

    is_emergency = ac_df['is_emergency'].any()
    line_weight  = 4 if is_emergency else 2.5

    first    = ac_df.iloc[0]
    last     = ac_df.iloc[-1]
    callsign = last['callsign']
    label    = f"{callsign} ({hex_id})" if callsign == 'UNK C/S' else callsign
    ac_type  = last['type_display']

    # Route into the correct type FeatureGroup
    fg = type_groups.get(ac_type, type_groups.get('Unknown'))

    alt_vals  = alt_series.dropna()
    alt_range = f"{int(alt_vals.min()):,} – {int(alt_vals.max()):,} ft" if not alt_vals.empty else 'N/A'
    first_alt = f"{int(alt_vals.iloc[0]):,} ft"  if not alt_vals.empty else 'N/A'
    last_alt  = f"{int(alt_vals.iloc[-1]):,} ft" if not alt_vals.empty else 'N/A'

    gs_vals  = gs_series.dropna()
    gs_range = f"{int(gs_vals.min())} – {int(gs_vals.max())} kts" if not gs_vals.empty else 'N/A'

    emergency_badge = f'<br><b style="color:red">⚠ {last["emergency_type"]}</b>' if is_emergency else ''

    popup_html = f"""
    <div style="font-family: monospace; min-width: 210px; font-size: 12px">
        <b style="font-size: 14px">{label}</b>{emergency_badge}<br><br>
        <b>Type:</b> {ac_type}<br>
        <b>Tail #:</b> {last['tail number'] or '—'}<br>
        <b>Hex:</b> {hex_id}<br>
        <b>Squawk:</b> {last['squawk']}<br>
        <b>Altitude:</b> {alt_range}<br>
        <b>Speed:</b> {gs_range}<br>
        <b>Source:</b> {last.get('source', '—')}<br>
        <b>Fixes:</b> {len(ac_df)}<br>
        <b>First:</b> {str(ac_df['datetime'].min())[:19]}<br>
        <b>Last:</b> {str(ac_df['datetime'].max())[:19]}
    </div>
    """

    # One colored segment per consecutive fix pair
    for j in range(len(coords) - 1):
        avg_alt = seg_avg(alt_series, j)
        avg_gs  = seg_avg(gs_series,  j)
        alt_str = f"{int(avg_alt):,} ft" if avg_alt is not None else 'N/A'
        gs_str  = f"{int(avg_gs)} kts"   if avg_gs  is not None else 'N/A'
        seg_color   = '#ff0000' if is_emergency else altitude_color(avg_alt)
        seg_tooltip = f"{label} | {ac_type} | {alt_str} | {gs_str}"
        folium.PolyLine(
            locations=[coords[j], coords[j + 1]],
            color=seg_color,
            weight=line_weight,
            opacity=0.85,
            tooltip=seg_tooltip
        ).add_to(fg)

    # Start marker
    folium.CircleMarker(
        location=[first['latitude'], first['longitude']],
        radius=5,
        color='white',
        weight=1,
        fill=True,
        fill_color='green',
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"{label} | {ac_type} | {first_alt} — start"
    ).add_to(fg)

    # End marker
    folium.CircleMarker(
        location=[last['latitude'], last['longitude']],
        radius=5,
        color='white',
        weight=1,
        fill=True,
        fill_color='white',
        fill_opacity=0.9,
        tooltip=f"{label} | {ac_type} | {last_alt} — last known"
    ).add_to(fg)

# Add all type FeatureGroups to the map
for fg in type_groups.values():
    fg.add_to(m)

# --- Altitude legend ---
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: rgba(0,0,0,0.75); padding: 10px 14px; border-radius: 6px;
            font-family: monospace; font-size: 12px; color: white; line-height: 1.8;">
  <b>Altitude</b><br>
  <span style="color:#ffa500">&#9644;</span> 0 ft<br>
  <span style="color:#ffff00">&#9644;</span> 10,000 ft<br>
  <span style="color:#00ff00">&#9644;</span> 20,000 ft<br>
  <span style="color:#00ffff">&#9644;</span> 30,000 ft<br>
  <span style="color:#0000ff">&#9644;</span> 35,000 ft<br>
  <span style="color:#800080">&#9644;</span> 40,000 ft<br>
  <span style="color:#ff0000">&#9644;</span> 50,000 ft<br>
  <span style="color:#ffffff">&#9644;</span> Altitude unknown<br>
  <span style="color:#ff0000">&#9644;&#9644;</span> Emergency (thick)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

output_path = os.path.join(project_root, 'analysis', 'flight_map.html')
m.save(output_path)
print(f'Map saved → {output_path}')

m

Aircraft with position data: 50
  ≥2 fixes (will draw track): 43
  1 fix  (skipped):           7

Aircraft types in dataset (7):
  C-17A Globemaster: 23 aircraft
  C-32 AIR FORCE TWO: 1 aircraft
  C-5M Galaxy: 1 aircraft
  E-3G AWACS: 1 aircraft
  F/A-18 Hornet: 1 aircraft
  KC-135R Stratotanker: 15 aircraft
  KC-46A Pegasus: 8 aircraft
Map saved → /Users/jasonchristopher/Desktop/Code-Fellows/Projects/Aircraft_Tracker/analysis/flight_map.html
